# Week 1: Strategic Planning and Data Exploration in Logistics
**Logistics Data Analyst Internship**  
**Author:** Saurabh Kumar  
**Objective:** Define a realistic regional distribution logistics scenario, establish core Key Performance Indicators (KPIs), map the end-to-end data science methodology, and demonstrate initial data exploration with Python.

---

## 1. Logistics Scenario Description
The operational scenario simulates a multi-depot regional fulfillment and distribution network serving retail and e-commerce customer orders.
* **Fulfillment Hubs:** 5 major regional distribution warehouses (`WH-Central`, `WH-North`, `WH-East`, `WH-West`, `WH-South`).
* **Fleet Mix:** Light delivery vans, medium-duty small trucks, and heavy-duty long-haul transport.
* **Service Levels:** Same Day, First Class (1-2 days), Second Class (2-3 days), and Standard Shipping (3-5 days).
* **Operational Variables:** Route distance (km), shipment payload volume, regional fuel price index, and real-time transit weather conditions.
* **Business Challenges:** Mitigating delivery delays, curbing escalating transport expenses, and balancing fulfillment workload across facilities.

## 2. Key Performance Indicators (KPIs)
To monitor service reliability, cost efficiency, and profitability, five primary KPIs are defined:

1. **On-Time Delivery Rate (%):**  
   $$\text{OTD Rate} = \frac{\sum (\text{actual\_days} \le \text{scheduled\_days})}{N} \times 100$$
2. **Late Delivery Rate (%):**  
   $$\text{Late Rate} = \frac{\sum (\text{actual\_days} > \text{scheduled\_days})}{N} \times 100 = 100 - \text{OTD Rate}$$
3. **Average Delivery Time (Days):**  
   $$\overline{T}_{\text{delivery}} = \frac{1}{N} \sum_{i=1}^N \text{actual\_days}_i$$
4. **Transport Cost per Shipment ($):**  
   $$\overline{C}_{\text{transport}} = \frac{1}{N} \sum_{i=1}^N \text{transport\_cost}_i$$
5. **Average Order Profit ($):**  
   $$\overline{P}_{\text{order}} = \frac{1}{N} \sum_{i=1}^N (\text{order\_value}_i - \text{modeled\_logistics\_cost}_i)$$

## 3. Data Science Methodology & Analytical Roadmap
The 7-stage end-to-end lifecycle designed for this project:
1. **Data Collection & Ingestion:** Ingest order dispatches, warehouse logs, transit metrics, and external weather data.
2. **Data Cleaning & Quality Assessment:** Resolve missing coordinates, screen recording errors via IQR, and standardize scales.
3. **Exploratory Data Analysis & Visualization:** Uncover distribution skews, cross-regional bottlenecks, and operational correlations.
4. **Feature Engineering:** Calculate distance-to-weight ratios, route velocity proxies, and SLA risk buffers.
5. **Predictive Modeling:** Train supervised regression algorithms (Linear, Decision Trees, Random Forest, Gradient Boosting) to forecast delivery transit times.
6. **Prescriptive Optimization:** Formulate vehicle routing and warehouse load-balancing constraints based on model outputs.
7. **Deployment & Performance Monitoring:** Track forecast drift, OTD adherence, and cost per shipment.

## 4. Initial Data Exploration & KPI Calculation Prototype

In [1]:
import pandas as pd
import numpy as np

# Load the benchmark logistics dataset
data_path = "../../data/processed/logistics_cleaned.csv"
try:
    df = pd.read_csv(data_path)
except FileNotFoundError:
    df = pd.read_csv("../../simulated_logistics_dataset.csv")

print(f"Dataset successfully loaded! Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Dataset successfully loaded! Dimensions: 1500 rows x 18 columns


,order_id,region,shipping_mode,vehicle_type,warehouse,weather,distance_km,shipment_volume,fuel_price,delivery_time_days,scheduled_days,late_delivery,transport_cost,order_value,profit,cost_is_outlier,distance_scaled,volume_scaled
0,100001,West,Second Class,Mini Truck,WH-B,Clear,141.748163,28.493749,98.459058,2.775334,3.0,0,2251.503544,2031.076646,622.67,1,0.006222,-0.241920
1,100002,East,Same Day,Truck,WH-A,Clear,118.500879,40.272770,105.472271,1.743251,1.0,1,2251.503544,3012.072273,1768.80,1,-0.294569,0.345398
2,100003,Central,Standard,Van,WH-C,Clear,204.640837,18.185547,94.368445,3.368115,5.0,0,2251.503544,2137.516262,596.09,1,0.819973,-0.755902
3,100004,West,Standard,Truck,WH-B,Rain,85.716277,29.051444,99.305041,3.324719,5.0,0,2251.503544,2313.341351,1117.14,1,-0.718760,-0.214113
4,100005,North,First Class,Van,WH-D,Cloudy,160.451514,28.067041,101.268911,1.609597,2.0,0,2659.739766,1958.744356,555.66,1,0.248219,-0.263197


### 4.1 Dataset Structure & Data Types

In [2]:
# Inspect data types, nulls, and memory usage
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            1500 non-null   int64  
 1   region              1500 non-null   object 
 2   shipping_mode       1500 non-null   object 
 3   vehicle_type        1500 non-null   object 
 4   warehouse           1500 non-null   object 
 5   weather             1500 non-null   object 
 6   distance_km         1500 non-null   float64
 7   shipment_volume     1500 non-null   float64
 8   fuel_price          1500 non-null   float64
 9   delivery_time_days  1500 non-null   float64
 10  scheduled_days      1500 non-null   float64
 11  late_delivery       1500 non-null   int64  
 12  transport_cost      1500 non-null   float64
 13  order_value         1500 non-null   float64
 14  profit              1500 non-null   float64
 15  cost_is_outlier     1500 non-null   int64  
 16  distan

### 4.2 Baseline KPI Performance

In [3]:
# Operational KPI Computations
on_time_rate = (df["delivery_time_days"] <= df["scheduled_days"]).mean() * 100
late_delivery_rate = 100 - on_time_rate
avg_delivery_time = df["delivery_time_days"].mean()
avg_transport_cost = df["transport_cost"].mean()
avg_order_value = df["order_value"].mean()
avg_profit = df["profit"].mean()

kpi_summary = pd.DataFrame({
    "Key Performance Indicator (KPI)": [
        "On-Time Delivery Rate (%)",
        "Late Delivery Rate (%)",
        "Average Delivery Time (Days)",
        "Average Transport Cost ($)",
        "Average Order Value ($)",
        "Average Profit per Order ($)"
    ],
    "Baseline Value": [
        f"{on_time_rate:.2f}%",
        f"{late_delivery_rate:.2f}%",
        f"{avg_delivery_time:.2f} days",
        f"${avg_transport_cost:.2f}",
        f"${avg_order_value:.2f}",
        f"${avg_profit:.2f}"
    ],
    "Target / Benchmark": [
        "> 85.00%",
        "< 15.00%",
        "< 3.00 days",
        "< $1,100.00",
        "> $2,000.00",
        "> $1,000.00"
    ]
})

kpi_summary

,Key Performance Indicator (KPI),Baseline Value,Target / Benchmark
0,On-Time Delivery Rate (%),56.00%,> 85.00%
1,Late Delivery Rate (%),44.00%,< 15.00%
2,Average Delivery Time (Days),3.56 days,< 3.00 days
3,Average Transport Cost ($),$1248.97,"< $1,100.00"
4,Average Order Value ($),$2332.97,"> $2,000.00"
5,Average Profit per Order ($),$977.08,"> $1,000.00"


### 4.3 Regional Operational Breakdown

In [4]:
# Segmenting logistics metrics across operating territories
regional_kpis = df.groupby("region").agg(
    total_shipments=("order_id", "count"),
    late_rate_pct=("late_delivery", lambda x: f"{x.mean()*100:.1f}%"),
    avg_delivery_days=("delivery_time_days", "mean"),
    avg_distance_km=("distance_km", "mean"),
    avg_cost=("transport_cost", "mean")
).reset_index()

regional_kpis.sort_values(by="avg_cost", ascending=False)

,region,total_shipments,late_rate_pct,avg_delivery_days,avg_distance_km,avg_cost
1,East,273,63.4%,4.004124,153.315648,1300.098683
0,Central,307,44.6%,3.577636,142.761833,1248.618154
4,West,323,31.6%,3.406304,137.526150,1247.497577
2,North,322,34.5%,3.294249,135.331003,1231.843733
3,South,275,49.8%,3.577662,138.983251,1220.404033


## 5. Strategic Conclusion & Next Steps
- **Reliability Gap:** With an initial on-time delivery rate of 56.00% and a late delivery rate of 44.00%, delivery reliability requires significant operational interventions.
- **Cost Impact:** Average transport cost stands at $1,242.32 per shipment, with clear variations across regions and distances.
- **Roadmap Progression:** Week 2 will implement the data collection, cleaning, and preprocessing pipeline to handle missing values, screen outliers, and scale features for machine learning.